In [1]:
"""
拖船调度软约束实例生成器 (Soft-Constraint Instance Generator)
支持生成不同难度级别的实例 - 索引修正版
"""

import numpy as np
import json
import os
from typing import Dict, List, Tuple
import matplotlib.pyplot as plt


class SoftConstraintInstanceGenerator:
    """软约束实例生成器"""
    
    def __init__(self, seed: int = 42):
        """初始化生成器"""
        self.seed = seed
        self.instances = {}
        np.random.seed(seed)
    
    def generate_instance(self,
                         num_tasks: int,
                         num_tugs: int,
                         instance_name: str,
                         difficulty: str = 'medium',
                         area_size: float = 30.0,
                         T_max: float = 24.0,
                         v_eco: float = 10.0) -> Dict:
        """生成单个实例"""
        
        print(f"生成: {instance_name:25s} 难度:{difficulty:8s} {num_tasks:3d}任务/{num_tugs:2d}拖船")
        
        # 获取难度参数
        diff_params = self._get_difficulty_params(difficulty, num_tasks)
        
        instance = {
            'metadata': {
                'name': instance_name,
                'num_tasks': num_tasks,
                'num_tugs': num_tugs,
                'area_size': area_size,
                'T_max': T_max,
                'v_eco': v_eco,
                'M': 1000.0,
                'W': 10000.0,
                'difficulty': difficulty,
                'expected_completion_rate': diff_params['expected_rate']
            },
            'base_location': [0.0, 0.0]
        }
        
        # 生成任务
        instance['tasks'] = self._generate_tasks(
            num_tasks, area_size, T_max, v_eco, diff_params
        )
        
        # 生成拖船
        instance['tugboats'] = self._generate_tugboats(
            num_tugs, num_tasks, diff_params
        )
        
        # 计算距离和时间矩阵
        instance['distance_matrix'] = self._compute_distance_matrix(instance)
        instance['time_matrix'] = self._compute_time_matrix(instance)
        
        self.instances[instance_name] = instance
        
        return instance
    
    def _get_difficulty_params(self, difficulty: str, num_tasks: int) -> Dict:
        """根据难度级别返回生成参数"""
        
        params = {
            'easy': {
                'multi_tug_prob': 0.15 if num_tasks <= 20 else 0.10,
                'time_window_width': 3.5,
                'fuel_multiplier': 2.5,
                'service_time_range': (0.8, 1.3),
                'expected_rate': '95-100%'
            },
            'medium': {
                'multi_tug_prob': 0.25 if num_tasks <= 20 else 0.15,
                'time_window_width': 2.5,
                'fuel_multiplier': 1.8,
                'service_time_range': (0.8, 1.6),
                'expected_rate': '85-95%'
            },
            'hard': {
                'multi_tug_prob': 0.35 if num_tasks <= 20 else 0.25,
                'time_window_width': 2.0,
                'fuel_multiplier': 1.3,
                'service_time_range': (1.0, 1.8),
                'expected_rate': '70-85%'
            },
            'extreme': {
                'multi_tug_prob': 0.45 if num_tasks <= 20 else 0.35,
                'time_window_width': 1.5,
                'fuel_multiplier': 1.1,
                'service_time_range': (1.2, 2.0),
                'expected_rate': '50-70%'
            }
        }
        
        return params.get(difficulty, params['medium'])
    
    def _generate_tasks(self, 
                       num_tasks: int,
                       area_size: float,
                       T_max: float,
                       v_eco: float,
                       diff_params: Dict) -> Dict:
        """生成任务数据"""
        
        tasks = {}
        task_types = ['靠泊', '离泊']
        
        total_time_span = T_max - 2.0
        
        for s in range(1, num_tasks + 1):
            # 任务类型
            task_type = np.random.choice(task_types)
            
            # 生成位置
            start_loc = np.random.uniform(-area_size/2, area_size/2, 2)
            angle = np.random.uniform(0, 2*np.pi)
            distance = np.random.uniform(3.0, 8.0)
            end_loc = start_loc + distance * np.array([np.cos(angle), np.sin(angle)])
            
            # 服务时长
            service_time = np.random.uniform(*diff_params['service_time_range'])
            
            # 时间窗
            time_slot = total_time_span / num_tasks
            earliest_start = 0.5 + (s - 1) * time_slot
            
            time_window_width = diff_params['time_window_width']
            max_latest_start = T_max - service_time - 0.5
            
            latest_start = min(earliest_start + time_window_width, max_latest_start)
            earliest_start = max(0.5, min(earliest_start, max_latest_start - 1.0))
            
            if latest_start <= earliest_start:
                latest_start = max_latest_start
                earliest_start = max(0.5, latest_start - time_window_width)
            
            # 需要的拖船数
            if np.random.random() < diff_params['multi_tug_prob']:
                num_tugs_needed = np.random.choice([2, 3], p=[0.8, 0.2])
            else:
                num_tugs_needed = 1
            
            # 最小马力需求
            if num_tugs_needed == 1:
                min_horsepower = np.random.randint(3000, 4500)
            elif num_tugs_needed == 2:
                min_horsepower = np.random.randint(6000, 9000)
            else:
                min_horsepower = np.random.randint(10000, 13000)
            
            # ✅ 使用字符串键与 load_data 保持一致
            tasks[str(s)] = {
                'type': task_type,
                'start_loc': start_loc.tolist(),
                'end_loc': end_loc.tolist(),
                'time_window': [round(earliest_start, 2), round(latest_start, 2)],
                'service_time': round(service_time, 2),
                'num_tugs_needed': int(num_tugs_needed),
                'min_horsepower': int(min_horsepower)
            }
        
        return tasks
    
    def _generate_tugboats(self, num_tugs: int, num_tasks: int, diff_params: Dict) -> Dict:
        """生成拖船数据"""
        
        tugboats = {}
        tug_names = ['Alpha', 'Beta', 'Gamma', 'Delta', 'Epsilon', 'Zeta', 
                     'Eta', 'Theta', 'Iota', 'Kappa', 'Lambda', 'Mu',
                     'Nu', 'Xi', 'Omicron', 'Pi', 'Rho', 'Sigma',
                     'Tau', 'Upsilon', 'Phi', 'Chi', 'Psi', 'Omega']
        
        for k in range(1, num_tugs + 1):
            # 马力
            horsepower = np.random.randint(4500, 6001)
            
            # 燃油容量
            base_fuel = 12000 if num_tasks <= 30 else (18000 if num_tasks <= 80 else 25000)
            fuel_capacity = int(base_fuel * diff_params['fuel_multiplier'])
            
            # 燃油系数
            alpha = round(np.random.uniform(0.15, 0.18), 3)
            beta = round(np.random.uniform(0.06, 0.08), 3)
            
            # ✅ 使用字符串键与 load_data 保持一致
            tugboats[str(k)] = {
                'name': tug_names[(k-1) % len(tug_names)],
                'horsepower': int(horsepower),
                'fuel_capacity': fuel_capacity,
                'alpha': alpha,
                'beta': beta
            }
        
        return tugboats
    
    def _compute_distance_matrix(self, instance: Dict) -> Dict:
        """
        计算距离矩阵 (稀疏字典格式)
        
        返回字典格式，键为 'i_j'，值为距离
        - '0_j': 基地 → 任务j入口
        - 'i_j': 任务i出口 → 任务j入口 (i≠j)
        - 'i_{n+1}': 任务i出口 → 基地
        """
        
        tasks = instance['tasks']
        base = np.array(instance['base_location'])
        n = len(tasks)
        
        distance_matrix = {}
        
        # 1. 基地(0) → 任务入口(1~n)
        for j in range(1, n + 1):
            start_loc = np.array(tasks[str(j)]['start_loc'])  # ✅ 使用字符串键
            distance_matrix[f'0_{j}'] = float(np.linalg.norm(base - start_loc))
        
        # 2. 任务出口(1~n) → 任务入口(1~n)
        for i in range(1, n + 1):
            end_i = np.array(tasks[str(i)]['end_loc'])  # ✅ 使用字符串键
            for j in range(1, n + 1):
                if i != j:
                    start_j = np.array(tasks[str(j)]['start_loc'])  # ✅ 使用字符串键
                    distance_matrix[f'{i}_{j}'] = float(np.linalg.norm(end_i - start_j))
        
        # 3. 任务出口(1~n) → 基地(n+1)
        for i in range(1, n + 1):
            end_loc = np.array(tasks[str(i)]['end_loc'])  # ✅ 使用字符串键
            distance_matrix[f'{i}_{n+1}'] = float(np.linalg.norm(end_loc - base))
        
        return distance_matrix
    
    def _compute_time_matrix(self, instance: Dict) -> Dict:
        """
        计算时间矩阵 (稀疏字典格式)
        
        time[i_j] = distance[i_j] / v_eco
        """
        
        v_eco = instance['metadata']['v_eco']
        distance_matrix = instance['distance_matrix']
        
        time_matrix = {}
        for key, distance in distance_matrix.items():
            time_matrix[key] = round(distance / v_eco, 4)
        
        return time_matrix
    
    def generate_all_instances(self):
        """生成128个不同难度的实例"""
        
        print("\n" + "🚢 " * 20)
        print("软约束拖船调度实例生成器 - 128个实例（分4个难度）")
        print("🚢 " * 20 + "\n")
        
        instance_count = 0
        difficulties = ['easy', 'medium', 'hard', 'extreme']
        
        for difficulty in difficulties:
            print(f"\n{'='*80}")
            print(f"生成难度: {difficulty.upper()}")
            print(f"{'='*80}\n")
            
            # 小规模 (8个)
            for num_tasks, num_tugs, area in [(10, 5, 30), (15, 6, 35), (20, 8, 40), (30, 12, 45)]:
                for v in [1, 2]:
                    self.seed = 42 + instance_count
                    np.random.seed(self.seed)
                    prefix = 'X' if difficulty == 'extreme' else difficulty[0].upper()
                    self.generate_instance(
                        num_tasks=num_tasks,
                        num_tugs=num_tugs,
                        instance_name=f'{prefix}_{num_tasks}_{num_tugs}_v{v}',
                        difficulty=difficulty,
                        area_size=area
                    )
                    instance_count += 1
            
            # 中规模 (12个)
            for num_tasks, num_tugs, area in [(40, 15, 50), (60, 20, 60), (80, 25, 70)]:
                for v in [1, 2, 3, 4]:
                    self.seed = 42 + instance_count
                    np.random.seed(self.seed)
                    prefix = 'X' if difficulty == 'extreme' else difficulty[0].upper()
                    self.generate_instance(
                        num_tasks=num_tasks,
                        num_tugs=num_tugs,
                        instance_name=f'{prefix}_{num_tasks}_{num_tugs}_v{v}',
                        difficulty=difficulty,
                        area_size=area
                    )
                    instance_count += 1
            
            # 大规模 (12个)
            for num_tasks, num_tugs, area in [(100, 30, 80), (150, 40, 100), (200, 50, 120)]:
                for v in [1, 2, 3, 4]:
                    self.seed = 42 + instance_count
                    np.random.seed(self.seed)
                    prefix = 'X' if difficulty == 'extreme' else difficulty[0].upper()
                    self.generate_instance(
                        num_tasks=num_tasks,
                        num_tugs=num_tugs,
                        instance_name=f'{prefix}_{num_tasks}_{num_tugs}_v{v}',
                        difficulty=difficulty,
                        area_size=area
                    )
                    instance_count += 1
        
        print(f"\n{'='*80}")
        print(f"总计生成 {instance_count} 个实例")
        print(f"{'='*80}")
    
    def save_instances(self, output_dir: str = 'instances_soft'):
        """保存所有实例到JSON文件"""
        
        os.makedirs(output_dir, exist_ok=True)
        
        print(f"\n{'='*80}")
        print(f"保存实例到目录: {output_dir}")
        print(f"{'='*80}\n")
        
        for name, instance in self.instances.items():
            filepath = os.path.join(output_dir, f"{name}.json")
            with open(filepath, 'w', encoding='utf-8') as f:
                json.dump(instance, f, indent=2, ensure_ascii=False)
            print(f"✓ 保存: {filepath}")
        
        # 保存汇总信息
        summary = {
            'total_instances': len(self.instances),
            'difficulties': {
                'easy': len([n for n in self.instances if n.startswith('E_')]),
                'medium': len([n for n in self.instances if n.startswith('M_')]),
                'hard': len([n for n in self.instances if n.startswith('H_')]),
                'extreme': len([n for n in self.instances if n.startswith('X_')])
            },
            'instance_names': list(self.instances.keys())
        }
        
        summary_path = os.path.join(output_dir, 'summary.json')
        with open(summary_path, 'w', encoding='utf-8') as f:
            json.dump(summary, f, indent=2, ensure_ascii=False)
        
        print(f"\n✓ 保存汇总信息: {summary_path}")


def main():
    """主函数"""
    
    generator = SoftConstraintInstanceGenerator(seed=42)
    generator.generate_all_instances()
    generator.save_instances(output_dir='instances_soft')
    
    print("\n" + "="*80)
    print("实例生成完成！")
    print("="*80)
    print(f"\n所有 128 个实例已保存到 'instances_soft' 目录")
    print(f"\n难度分布:")
    print(f"  Easy (E_):    32个 - 预期完成率 95-100%")
    print(f"  Medium (M_):  32个 - 预期完成率 85-95%")
    print(f"  Hard (H_):    32个 - 预期完成率 70-85%")
    print(f"  Extreme (X_): 32个 - 预期完成率 50-70%")


if __name__ == "__main__":
    main()


🚢 🚢 🚢 🚢 🚢 🚢 🚢 🚢 🚢 🚢 🚢 🚢 🚢 🚢 🚢 🚢 🚢 🚢 🚢 🚢 
软约束拖船调度实例生成器 - 128个实例（分4个难度）
🚢 🚢 🚢 🚢 🚢 🚢 🚢 🚢 🚢 🚢 🚢 🚢 🚢 🚢 🚢 🚢 🚢 🚢 🚢 🚢 


生成难度: EASY

生成: E_10_5_v1                 难度:easy      10任务/ 5拖船
生成: E_10_5_v2                 难度:easy      10任务/ 5拖船
生成: E_15_6_v1                 难度:easy      15任务/ 6拖船
生成: E_15_6_v2                 难度:easy      15任务/ 6拖船
生成: E_20_8_v1                 难度:easy      20任务/ 8拖船
生成: E_20_8_v2                 难度:easy      20任务/ 8拖船
生成: E_30_12_v1                难度:easy      30任务/12拖船
生成: E_30_12_v2                难度:easy      30任务/12拖船
生成: E_40_15_v1                难度:easy      40任务/15拖船
生成: E_40_15_v2                难度:easy      40任务/15拖船
生成: E_40_15_v3                难度:easy      40任务/15拖船
生成: E_40_15_v4                难度:easy      40任务/15拖船
生成: E_60_20_v1                难度:easy      60任务/20拖船
生成: E_60_20_v2                难度:easy      60任务/20拖船
生成: E_60_20_v3                难度:easy      60任务/20拖船
生成: E_60_20_v4                难度:easy      60任务/20拖船
生成: E_80_25_v1            